In [1]:
# imports
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    mean_squared_error, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report, 
    roc_curve, 
    roc_auc_score
)
from sklearn.utils import resample
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# function to filter dataframe
def filter_df(df):
    
    # only originated loans
    df = df[(df['action_taken'] == 1)]
    
    # only home purchases
    df = df[(df['loan_purpose'] == 1)]
    
    # not commerical/business purpose
    df = df[(df['business_or_commercial_purpose'] == 2)]
    
    # only principal dwelling
    df = df[(df['occupancy_type'] ==  1)]
    
    # only Illinois
    df = df[(df['state_code'] ==  'IL')]
    
    return df

In [3]:
# function to remove columns that aren't necessary
def remove_columns(df):
    
    # columns to drop
    columns_to_drop = ['activity_year', 'lei', 'derived_msa-md', 'county_code', 'state_code', 
                       'census_tract', 'loan_purpose', 'business_or_commercial_purpose', 
                       'occupancy_type', 'denial_reason-1', 'denial_reason-2', 'denial_reason-3', 'denial_reason-4',
                       'action_taken']
    existing_columns = df.columns.tolist() 
    columns_to_drop = [col for col in columns_to_drop if col in existing_columns]
    df = df.drop(columns=columns_to_drop)
    return df

In [4]:
# function to remove columns with high nan values
def remove_nan_columns(df, threshold=50.0):
    nan_percentage = df.isna().mean() * 100
    columns_to_drop = nan_percentage[nan_percentage > threshold].index.tolist()
    df = df.drop(columns=columns_to_drop)
    return df

In [5]:
# function to combine races
def combine_race_ethnicity(df):
    for i in range(1, 6):
        ethnicity_col = f'applicant_ethnicity-{i}'
        if ethnicity_col in df.columns:
            df[ethnicity_col] = df[ethnicity_col].apply(lambda x: 1 if x in [11, 12, 13, 14] else x)
        
        co_ethnicity_col = f'co-applicant_ethnicity-{i}'
        if co_ethnicity_col in df.columns:
            df[co_ethnicity_col] = df[co_ethnicity_col].apply(lambda x: 1 if x in [11, 12, 13, 14] else x)
        
        race_col = f'applicant_race-{i}'
        if race_col in df.columns:
            df[race_col] = df[race_col].apply(lambda x: 2 if x in [21, 22, 23, 24, 25, 26, 27] else (4 if x in [41, 42, 43, 44] else x))
        
        co_race_col = f'co-applicant_race-{i}'
        if co_race_col in df.columns:
            df[co_race_col] = df[co_race_col].apply(lambda x: 2 if x in [21, 22, 23, 24, 25, 26, 27] else (4 if x in [41, 42, 43, 44] else x))
            
        if 'derived_ethnicity' in df.columns:
            df['derived_ethnicity'] = df['derived_ethnicity'].apply(lambda x: 'Hispanic or Latino' if x == 'Free Form Text Only' else x)
    return df

In [6]:
# function to remove outliers through the IQR method
def iqr_outliers(df):
    
    # all the columns are numerical 
    columns = ['loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread', 
           'total_loan_costs', 'total_points_and_fees', 'origination_charges', 'discount_points', 
           'lender_credits', 'loan_term', 'prepayment_penalty_term', 'intro_rate_period', 'property_value', 
           'multifamily_affordable_units', 'income', 'tract_population', 'tract_minority_population_percent', 
           'ffiec_msa_md_median_family_income', 'tract_to_msa_income_percentage', 'tract_owner_occupied_units',
           'tract_one_to_four_family_homes', 'tract_median_age_of_housing_units']
    
    # apply to numerical columns
    iqr_factor = 1.5
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - iqr_factor * IQR
            upper_bound = Q3 + iqr_factor * IQR
            outlier_indices = (df[col] < lower_bound) | (df[col] > upper_bound)
            df = df[~outlier_indices]
            
    return df

In [7]:
# function that combines all previous functions
def clean(df, year):
    clean_df = iqr_outliers(combine_race_ethnicity(remove_nan_columns(remove_columns(filter_df(df)))))
    clean_df.to_csv(f'clean_{year}_rs.csv', index=False)

In [8]:
#2018
hmda_2018 = pd.read_csv('actions_taken_1-3_loan_purposes_1_year_2018.csv')
clean(hmda_2018, '2018')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_78692/1086300646.py:2: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2018 = pd.read_csv('actions_taken_1-3_loan_purposes_1_year_2018.csv')


In [9]:
#2019
hmda_2019 = pd.read_csv('actions_taken_1-3_year_2019_loan_purposes_1.csv')
clean(hmda_2019, '2019')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_78692/454767756.py:2: DtypeWarning: Columns (5,22,23,24,26,27,28,29,30,31,32,33,38,43,44,78,80) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2019 = pd.read_csv('actions_taken_1-3_year_2019_loan_purposes_1.csv')


In [10]:
#2020
hmda_2020 = pd.read_csv('actions_taken_1-3_loan_purposes_1_year_2020.csv')
clean(hmda_2020, '2020')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_78692/488139666.py:2: DtypeWarning: Columns (5,22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2020 = pd.read_csv('actions_taken_1-3_loan_purposes_1_year_2020.csv')


In [11]:
#2021
hmda_2021 = pd.read_csv('actions_taken_1-3_loan_purposes_1_year_2021.csv')
clean(hmda_2021, '2021')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_78692/1772294265.py:2: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2021 = pd.read_csv('actions_taken_1-3_loan_purposes_1_year_2021.csv')


In [12]:
#2022
hmda_2022 = pd.read_csv('actions_taken_1-3_year_2022_loan_purposes_1.csv')
clean(hmda_2022, '2022')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_78692/628867052.py:2: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2022 = pd.read_csv('actions_taken_1-3_year_2022_loan_purposes_1.csv')
